In [2]:
import xarray as xr
import numpy as np

import sys
sys.path.append('../new_model')
from constants import g,R_dry
from physics_equations import specific_humidity_from_rh


In [3]:
#load the datasets -radiometer profiles

cabauw_radiometer = xr.open_mfdataset(r'..\data\Cabaw_multi_radiometer\*.nc')
time = cabauw_radiometer.time

In [4]:
#load surface data (cesar network)


cabauw_surface = xr.open_mfdataset(r'..\data\Cabauw_cesar_surface\cesar_surface_meteo_lc1_t10_v1.0_202407.nc')

cabauw_surface = cabauw_surface.sel(time=time,method='nearest')

vars = ['TA002','P0','TD002','RH002']
surface_vars = cabauw_surface[vars]

T_surf = surface_vars.TA002
p_surf = surface_vars.P0*100
T_d = surface_vars.TD002
rh_surf = surface_vars.RH002/100


In [5]:
#combining the datasets so that the first value of each profile corresponds to the surface

cabauw_radiometer = cabauw_radiometer.assign_coords(height=[2 if h == cabauw_radiometer.height[0] else h for h in cabauw_radiometer.height.values])
radiometer_surface = cabauw_radiometer.height[0]
cabauw_radiometer.temperature.loc[dict(height=radiometer_surface)] = T_surf.values
cabauw_radiometer.relative_humidity.loc[dict(height=radiometer_surface)] = rh_surf.values


In [6]:
#extract the data
my_temp = cabauw_radiometer.temperature.values
Z = cabauw_radiometer.height.values
T_d = T_d.values

In [7]:
#build the pressure profile
p = np.zeros(cabauw_radiometer.temperature.shape)
p[:,0] = p_surf #Pa

for i in range(1,cabauw_radiometer.height.shape[0]):
    temp_mean = np.mean(my_temp[:, 0:i+1], axis=1)  # shape: (n_times,)
    height_i = cabauw_radiometer.height[i].values   # make it a NumPy scalar
    exponent = g * height_i / (R_dry * temp_mean)   # now safe elementwise op
    p[:, i] = p[:, 0] / np.exp(exponent)

In [8]:
#build the specific humidity profile
RH_env = cabauw_radiometer.relative_humidity.values
RH_env[RH_env > 1] = 1
q_v_air = specific_humidity_from_rh(my_temp, RH_env, p)

#### Saving the data

In [9]:
time = cabauw_radiometer.time

# Create xarray Dataset
ds_profiles = xr.Dataset(
    {
        "temperature": (["time", "height"], my_temp),
        "pressure": (["time", "level"], p),
        "specific_humidity": (["time", "height"], q_v_air),
        "dew_point_temperature": (["time"], T_d)
    },
    coords={
        "time": time,
        "height": Z
    }
)

# Add metadata
ds_profiles["temperature"].attrs["units"] = "K"
ds_profiles["pressure"].attrs["units"] = "Pa"
ds_profiles["specific_humidity"].attrs["units"] = "kg/kg"
ds_profiles["dew_point_temperature"].attrs["units"] = "K"

# Save to NetCDF
ds_profiles.to_netcdf(r"..\data\model_inputs\Cabauw_radiometer\radiometer_profile_data.nc")